In [20]:
import torch
import torch.nn as nn

import pandas as pd
import numpy as np

import inspect

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(PROJECT_ROOT)

c:\Users\kakas\Documents\Projects\wqm_final\WQM_AI


In [21]:
df = pd.read_csv("../dataset/generated/water_quality_dataset.csv")

df.head()

,Timestamp,pH,Temperature,Dissolved_Oxygen,Turbidity,Water_Quality
0,2026-01-01 00:00:00,7.21,26.90,8.21,24783.23,Good
1,2026-01-01 00:00:05,7.22,26.63,8.20,8674.25,Good
2,2026-01-01 00:00:10,7.21,26.85,8.19,3035.97,Good
3,2026-01-01 00:00:15,7.22,27.17,8.18,1062.35,Good
4,2026-01-01 00:00:20,7.19,27.07,8.17,371.82,Good


In [22]:
df["Water_Quality"] = df["Water_Quality"].map({

    "Good":1,

    "Bad":0

})

In [23]:
X = df[

    [

        "pH",

        "Temperature",

        "Dissolved_Oxygen",

        "Turbidity"

    ]

].values

y = df["Water_Quality"].values

In [24]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [25]:
def create_sequences(X, y, sequence_length=10):

    X_seq = []
    y_seq = []

    for i in range(len(X) - sequence_length):

        X_seq.append(X[i:i+sequence_length])

        y_seq.append(y[i+sequence_length])

    return np.array(X_seq), np.array(y_seq)

In [26]:
SEQUENCE_LENGTH = 10

X, y = create_sequences(
    X,
    y,
    sequence_length=SEQUENCE_LENGTH
)

In [27]:
print(X.shape)
print(y.shape)

(99990, 10, 4)
(99990,)


In [28]:
X_train,X_test,y_train,y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    shuffle=True

)

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(device)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1).to(device)

Using Device: cuda


In [30]:
import torch.optim as optim

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

from src.model.nflnn import NFLNN
from src.model.trainer import Trainer

In [31]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

In [32]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [33]:
model = NFLNN().to(device)

print(model)

NFLNN(
  (membership): GaussianMembership()
  (rules): FuzzyRuleLayer(
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (rule_generator): Sequential(
      (0): Linear(in_features=12, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
      (3): ReLU()
    )
  )
  (liquid): LiquidCell(
    (input_layer): Linear(in_features=32, out_features=32, bias=True)
    (hidden_layer): Linear(in_features=32, out_features=32, bias=True)
    (tau_layer): Linear(in_features=32, out_features=32, bias=True)
    (activation): Tanh()
  )
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
    (4): Sigmoid()
  )
)


In [34]:
criterion = nn.BCELoss()

In [35]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

AttributeError: module 'torch' has no attribute '_utils'

In [ ]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device
)

NameError: name 'optimizer' is not defined

In [ ]:
OUTPUT_DIR = "outputs"

MODEL_DIR = os.path.join(OUTPUT_DIR, "models")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

for folder in [MODEL_DIR, LOG_DIR, PLOT_DIR, REPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Output folders created successfully.")

Output folders created successfully.


In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": []
}

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: torch.Size([79992, 10, 4])
y_train shape: torch.Size([79992, 1])


In [ ]:
x, y = next(iter(train_loader))

print("Batch X shape:", x.shape)
print("Batch y shape:", y.shape)

print(inspect.getsource(NFLNN.forward))

Batch X shape: torch.Size([64, 10, 4])
Batch y shape: torch.Size([64, 1])
    def forward(self, x):

        batch_size = x.size(0)

        h = torch.zeros(
            batch_size,
            self.liquid.hidden_dim,
            device=x.device 
        )

        sequence_length = x.size(1)

        for t in range(sequence_length):

            xt = x[:, t, :]

            memberships = self.membership(xt)

            rules = self.rules(memberships)

            h = self.liquid(
                rules,
                h)

        output = self.classifier(h)

        return output



In [ ]:
output = model(x.to(device))

print("Model Output Shape:", output.shape)

Model Output Shape: torch.Size([64, 1])


In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    train_loss, train_acc = trainer.train_epoch(train_loader)
    test_loss, test_acc = trainer.evaluate(test_loader)

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Acc: {train_acc:.4f} "
        f"| Test Loss: {test_loss:.4f} "
        f"| Test Acc: {test_acc:.4f}"
    )

Epoch [1/30] | Train Loss: 0.1924 | Train Acc: 0.9070 | Test Loss: 0.0765 | Test Acc: 0.9787
Epoch [2/30] | Train Loss: 0.0659 | Train Acc: 0.9786 | Test Loss: 0.0303 | Test Acc: 0.9932
Epoch [3/30] | Train Loss: 0.0353 | Train Acc: 0.9922 | Test Loss: 0.0241 | Test Acc: 0.9938
Epoch [4/30] | Train Loss: 0.0321 | Train Acc: 0.9929 | Test Loss: 0.0265 | Test Acc: 0.9937
Epoch [5/30] | Train Loss: 0.0313 | Train Acc: 0.9928 | Test Loss: 0.0234 | Test Acc: 0.9938
Epoch [6/30] | Train Loss: 0.0268 | Train Acc: 0.9934 | Test Loss: 0.0196 | Test Acc: 0.9953
Epoch [7/30] | Train Loss: 0.0265 | Train Acc: 0.9940 | Test Loss: 0.0249 | Test Acc: 0.9936
Epoch [8/30] | Train Loss: 0.0217 | Train Acc: 0.9939 | Test Loss: 0.0262 | Test Acc: 0.9947
Epoch [9/30] | Train Loss: 0.0219 | Train Acc: 0.9946 | Test Loss: 0.0278 | Test Acc: 0.9948
Epoch [10/30] | Train Loss: 0.0210 | Train Acc: 0.9949 | Test Loss: 0.0220 | Test Acc: 0.9947
Epoch [11/30] | Train Loss: 0.0208 | Train Acc: 0.9946 | Test Loss: 0

In [ ]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(LOG_DIR, "training_history.csv"),
    index=False
)

print("Training history saved.")

In [ ]:
torch.save(
    model.state_dict(),
    os.path.join(MODEL_DIR, "best_model.pth")
)

print("Model saved.")